# pseudo.ipynb — MRI→3D Deformation Transfer 설계 (수도코드)

> ⚠️ 실행용이 아니라 **설계 수도코드**입니다. 함수 내부는 스텁(`...`/TODO)이며, 흐름·좌표계·근거를 정리한 문서.

## 핵심 결정 (합의된 설계)
1. **A = 공간 정합(registration)** — MRI공간 → 모델공간. **rest 에서 딱 1회** 계산.
2. **delta = 프레임별 움직임(transfer)** — rest→target 변위. **모델공간에서** 계산.
3. **출력 = `ref`(3D 모델)를 변형한 mesh** — contour 재구성(`Contour2Mesh`/lift)이 아니라 **ref 를 변형**.
   → 토폴로지(433정점·면 순서) 보존 → **FEM DB 와 정점 대 정점 대응이 공짜**(최종 목표 §9).

## 하지 말 것
- `target_3D = src_contour * A * T` 처럼 **윤곽에 행렬을 곱해 끝내기**:
  (a) 출력이 mesh 가 아니라 윤곽, (b) 비강체 움직임을 행렬 하나로 못 담음, (c) 좌표계 섞임.
- `Contour2Mesh(target_3D)` 로 **윤곽에서 mesh 재구성**: lateral 가정 필요 + ref 3D 디테일·정점 대응 상실.

## 좌표계 표기
| 공간 | 뜻 |
|---|---|
| **image (px)** | MRI 원본 (row 아래+, col 오른쪽+) |
| **image-mm** | `x=col·mm_per_px`, `y=(H-1-row)·mm_per_px` (위쪽 +) |
| **model (mm)** | `x`=전후, `y`=좌우(≈0), `z`=위 |

`A` 가 잇는 것: **image-mm `(x,y)` → model `(x,z)`**.

## 0. 입력 로드

In [ ]:
src = load_mri(subject, REST_FRAME)     # MRI rest 프레임 (image)
ref = load_model(obj_path)              # 3D 모델: ref.verts (N,3) mm, ref.faces (M,3)
# tar(=target 프레임) 는 아래 프레임 루프에서 로드

## 1. 윤곽 추출
두 윤곽 모두 **tip 시작 · 같은 방향(CW)** 규약으로 정렬 → 나중에 점 대응이 맞음.

In [ ]:
src_contour = extract_contour(src)      # MRI rest 윤곽: image-mm (K,2), cw_from_tip
ref_contour = extract_contour3D(ref)    # 모델 정중시상 윤곽: model x,z (K',2), cw_from_tip
#   extract_contour3D = y=0 슬라이스(삼각형-평면 교차 체인). raw 정점 각도정렬은 엉킴(금지).

## 2. A = registration  — rest 에서 1회
`src_contour`(MRI) ↔ `ref_contour`(모델)을 **대응(correspond)** 시킨 뒤 affine/similarity fit.
> 3점 랜드마크만 쓰면 나머지가 안 붙음(rest gap ≈12mm 확인). → 랜드마크로 구간 대응 후 fit 권장.

In [ ]:
def calculate_registration(src_contour, ref_contour):
    # 1) 대응: tip·dorsum·root 앵커로 구간 분할 후 호길이 리샘플 (correspondence.py)
    s, r = correspond(src_contour, ref_contour)        # 같은 점수, k<->k 해부학 정렬
    # 2) fit: image-mm (x,y) -> model (x,z) 최적 affine/similarity (lstsq / Procrustes)
    A = fit_affine(s, r)                               # 반환: 적용가능한 변환 (+ 잔차)
    return A
    # ...

A = calculate_registration(src_contour, ref_contour)   # 이후 모든 프레임에 재사용

## 3. 프레임별 retarget (transfer + deform)
움직임은 **모델공간에서** 잰다: `src`, `tar` 윤곽을 **둘 다 A로 올린 뒤** 빼기.
→ A 의 선형부만 변위에 적용돼 translation 이중적용을 자동 회피.

In [ ]:
def retarget_frame(tar, ref, ref_contour, src_contour, A):
    tar_contour = extract_contour(tar)                 # MRI target 윤곽 (image-mm, cw_from_tip)

    # (a) 두 윤곽을 모델공간으로
    src_m = A(src_contour)                             # rest 윤곽 @ model
    tar_m = A(tar_contour)                             # target 윤곽 @ model

    # (b) 대응 맞춘 뒤 변위(delta) — ref_contour 위치 기준
    s = correspond(src_m, ref_contour)                 # (n,2)
    t = correspond(tar_m, ref_contour)                 # (n,2)
    delta = t - s                                      # (n,2) model 공간 변위
    #   (선택) delta 스무딩: 곡선따라(spatial) + 프레임축(temporal)

    # (c) RBF skinning: 윤곽 변위를 전체 정점으로 전파
    warp = RBF(centers=ref_contour, values=delta,      # gaussian, epsilon=1/rbf_len
               kernel="gaussian")
    d_xz = warp(ref.verts[:, [0, 2]])                  # 각 정점의 (x,z) 변위

    # (d) ref 를 변형 — x,z 만, y(좌우)는 그대로 → 토폴로지 보존
    deformed = ref.verts.copy()
    deformed[:, 0] += d_xz[:, 0]
    deformed[:, 2] += d_xz[:, 1]

    return Mesh(verts=deformed, faces=ref.faces)       # ← 출력은 mesh (Contour2Mesh 불필요)

## 4. 비디오(시퀀스) 구동
A 는 한 번, `retarget_frame` 은 프레임마다. rest 대비 상대 변위라 rest 프레임은 변형 0.

In [ ]:
def run(subject, frames):
    src = load_mri(subject, REST_FRAME)
    ref = load_model(obj_path)
    src_contour = extract_contour(src)
    ref_contour = extract_contour3D(ref)
    A = calculate_registration(src_contour, ref_contour)     # 1회

    meshes = []
    for f in frames:
        tar = load_mri(subject, f)
        meshes.append(retarget_frame(tar, ref, ref_contour, src_contour, A))
    return meshes                                            # list[Mesh], 전부 같은 토폴로지

## 5. 왜 이 형태인가 (요약)
- **A(공간)와 delta(시간)를 분리** → A 는 rest 1회, 움직임만 프레임별.
- **delta 는 곱이 아니라 변위장** → 비강체 움직임을 점별로 표현.
- **ref 를 변형(RBF), Contour2Mesh 아님** → 3D 해부 디테일 + 토폴로지 보존.
- **토폴로지 보존** → FEM 출력(같은 433정점)과 정점 대 정점 매칭 → activation 역추정(§9)로 직결.

## 남은 채울 곳 (구현 시)
- `correspond(...)` : `correspondence.py`(랜드마크 구간 대응) — 바닥 불일치 개선 진행 중.
- `fit_affine(...)` : affine/similarity/(비강체 TPS) 중 선택. rest 정합 품질이 전체 정확도의 상한.
- delta 스무딩 파라미터: `rbf_len`, `spatial_win`, `temporal_win`.
- lateral(y): 지금은 불변. 필요하면 FEM prior 로 채움(별도 트랙).